# Current Advances in LLM Reasoning — live demo notebook

The hands-on companion to the ACL 2026 tutorial ([llmreasoning.github.io](https://llmreasoning.github.io)). Each section is a short, runnable story:

1. **Instability** — do models even reason _consistently_?
2. **Test-time strategies** — spending inference compute to reason _better_.
3. **Post-training** — how SFT / preference / RL stages reshape one model.
4. **Budget forcing** — directly controlling how long the model thinks.
5. **Multilinguality** — does the model reason in the language you ask in?


In [ ]:
%load_ext autoreload
%autoreload 2
import asyncio
from demo.reasoning_demo.client import LLM
from demo.reasoning_demo import mgsm as M
from demo.reasoning_demo import traces as T   # loads the shipped s1.1 generations
from IPython.display import HTML, display
import demo.instability.consistency as C

import json
from IPython.display import display

from helpers import show, extract_boxed, think_span

## 1. Instability in LLM reasoning

Before we try to make models reason _better_, we should ask whether they reason _consistently_ at all. Across this section the model, the problem, and the settings stay fixed — only the sampling seed changes — and the answer moves anyway.


### 1.1 What's the instability in LLM reasoning

We ask the **same model** the **same question** with the **same settings**, twice in a row. Nothing changes between the two calls except the sampling randomness.

Insights:

- Same prompt, same decoding parameters. Yet the two runs can land on different answers (or reach the same one by different work).
- This isn't a bug: sampling makes generation stochastic. The question is how much it moves the _answer_, not just the wording.


In [ ]:
llm = LLM()

PROBLEM = 2   # which MGSM problem (0–249); the same index is the same problem in every language
prob_en = M.load_mgsm('en', n=PROBLEM + 1)[PROBLEM]


print('REQUEST:')
r_en = await M.asolve_direct(llm, prob_en.question, 'en')
print(r_en.detail['prompt'])
print('\nRESPONSE 1:')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'WRONG'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

print('\nRESPONSE 2:')
r_en = await M.asolve_direct(llm, prob_en.question, 'en')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'WRONG'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

### 1.2 How severe can the instability be?

The interactive Game 24 demo makes it concrete: one model, one puzzle, many identical runs, overlaid on a single tree. Run it live with the command below, or read the snapshot underneath.

Insights:

- The valid runs collapse onto one shared trunk; the failures peel off exactly where the model fabricates a number it was never given.
- Same puzzle, same settings. The spread of outcomes _is_ the instability, and made visual, looks significant.


`uv run python demo/serve.py --open game24 --no-strategies`


![Game 24](assets/game24.png)


### 1.3 How often this happens in practise?

Zooming out from a single puzzle to a whole grid: each model × benchmark cell is 30 reruns × 50 problems, drawn as a solved-or-not (0/1) heatmap.

Insights:

- A mottled rather than solid heatmap means the model flips between right and wrong across seeds on the _same_ problems.
- Instability is specific: a model can be flaky on one benchmark but steady on another, and a benchmark flaky for one model but not another.


In [48]:
MODEL_1 = 'gpt-oss-120b'
MODEL_2 = 'gemini-3-flash'
MODEL_3 = "gpt-4.1-mini"

BENCHMARK_1 = 'game24' 
BENCHMARK_2 = "scibench"

C.show_heatmap(MODEL_1,     BENCHMARK_1)

This doesn't mean that the _same model_ is unstable for all benchmarks


In [ ]:
C.show_heatmap(MODEL_1,     BENCHMARK_2)

The also doesn't mean that the _same benchmark_ is unstable for all models


In [ ]:
C.show_heatmap(MODEL_2,     BENCHMARK_1)

But it surely happens often.


In [ ]:
C.show_heatmap(MODEL_2,     "hle")
C.show_heatmap(MODEL_3,     "game24")
C.show_heatmap(MODEL_1,     "sonnetwriting")

### 1.4 Does instability matter?

If a model is right on some seeds and wrong on others, the single number a benchmark reports is a _choice_ of scoring protocol, not a fact about the model.

Insights:

- The same reruns scored as single-run / mean@N / maj@N / pass@N give very different "capability".
- maj@N (majority vote) and pass@N (any run succeeds) turn that flakiness into higher reported scores — drag N in the demo to watch it move.


`uv run python demo/serve.py --open protocols --no-strategies`


In [ ]:
C.show_protocols('game24')

## 2. Comparing budget utilization across reasoning strategies

One way to make a model reason _better_ is simply to spend more compute at inference time. Here we take a single GSM8K problem and solve it with seven test-time strategies, then compare how each one spends its budget.

Insights:

- The budget shows up three ways: **latency** (wall-clock), **tokens**, and **calls** (round-trips to the model).
- There's no single best strategy. The winner depends on the criteria you care most about such as latency, accuracy, ease of implementation, and generalizability.


`uv run python demo/serve.py --open strategies`


In [ ]:
# The notebook version of the interactive demo: pick one GSM8K problem and solve
# it with each test-time strategy, then compare how each spends its budget.
from demo.reasoning_demo.data import load_problems
from demo.reasoning_demo.extract import is_correct
from demo.reasoning_demo.strategies import (
    input_output, self_consistency, react, agentic,
    iterative, tree_of_thoughts, fleet_of_agents,
)

llm = LLM()
problems = load_problems() # bundled GSM8K sample
prob = next((p for p in problems if p.id == 'gsm8k-3'), problems[0])

print(f'Model: {llm.model}')
print(f'Problem [{prob.id}]:\n  {prob.question}')
print(f'Gold answer: {prob.answer}\n')

# same problem, seven ways to spend inference compute (sampling, searching, refining, tools)
methods = [
    ('input-output (direct)',  lambda: input_output(llm, prob.question)),
    ('self-consistency k=5',   lambda: self_consistency(llm, prob.question, k=5)),
    ('react (<=6 steps)',      lambda: react(llm, prob.question, max_steps=6)),
    ('agent (tool-use)',       lambda: agentic(llm, prob.question, max_steps=6)),
    ('self-refine (2 rounds)', lambda: iterative(llm, prob.question, rounds=2)),
    ('tree-of-thoughts d=2',   lambda: tree_of_thoughts(llm, prob.question, depth=2)),
    ('fleet-of-agents n=3',    lambda: fleet_of_agents(llm, prob.question, n_agents=4)),
]

print(f"{'method':<24}{'answer':>8}{'correct':>9}{'latency':>10}{'tokens':>9}{'calls':>7}")
print('-' * 67)
for label, run in methods:
    r = run()
    ok = 'yes' if is_correct(r.answer, prob.answer) else 'NO'
    print(f'{label:<24}{str(r.answer):>8}{ok:>9}'
          f'{r.latency_s:>9.2f}s{r.usage.total_tokens:>9}{r.usage.calls:>7}')

Model: meta-llama/llama-3.1-8b-instruct
Problem [gsm8k-3]:
  James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?
Gold answer: 540

method                    answer  correct   latency   tokens  calls
-------------------------------------------------------------------
input-output (direct)        540      yes     1.29s       90      1
self-consistency k=5         540      yes     6.97s     1061      5
react (<=6 steps)            540      yes    11.38s     3511      7
agent (tool-use)             540      yes     4.35s      966      2
self-refine (2 rounds)       540      yes     9.91s     2847      5
tree-of-thoughts d=2         540      yes     6.00s     2854     13
fleet-of-agents n=4          540      yes    10.74s     5198     25


## 3. Comparing the effects of different Post-training methods


We take one base model **InternVL3.5-1B** — and observe four _cumulative_ post-training stages, then ask the **same** question at each stage and watch the answer (and the reasoning behind it) change:

1. **Pretrained** — next-token prediction only, no alignment
2. **Pretrained + SFT** — supervised fine-tuning on instruction → response data
3. **Pretrained + SFT + MPO** — Mixed Preference Optimization (preference alignment)
4. **Pretrained + SFT + MPO + GSPO** — Group Sequence Policy Optimization (a final RL stage)

The task is a **MathVerse "Vision Only"** geometry problem — the question _and_ the answer choices are baked into the figure, so the image below is **all** the model sees.


In [ ]:
run = json.load(open('results/post_run.json'))
sample = run['sample']

# map each saved model id -> its post-training stage, and index the runs by stage
STAGE = {
    'OpenGVLab/InternVL3_5-1B-Pretrained': 'Pretrained',
    'OpenGVLab/InternVL3_5-1B-Instruct':   'Pretrained + SFT',
    'OpenGVLab/InternVL3_5-1B-MPO':        'Pretrained + SFT + MPO',
    'OpenGVLab/InternVL3_5-1B':            'Pretrained + SFT + MPO + GSPO',
}
runs = {STAGE[r['model']]: r for r in run['results']}



print(f"{run['dataset']['name']} · {sample['problem_version']} · "
      f"{sample['question_type']} · gold answer = {sample['gold_answer']}")

![MathVerse](assets/mathverse_sample.png)


### Stage 1 — Pretrained

Only next-token prediction on web-scale data, no alignment. It barely _answers_: it emits a short multiple-choice-looking string and stops.


In [ ]:
show(runs, 'Pretrained')

### Stage 2 — + SFT

Supervised fine-tuning teaches the **format** of a helpful answer. Now it produces a full step-by-step solution and commits to a final value.


In [ ]:
show(runs, 'Pretrained + SFT')

### Stage 3 — + MPO

Preference optimization tunes **which** answers the model favours. The solution stays structured but gets tighter and more direct.


In [ ]:
show(runs, 'Pretrained + SFT + MPO')

### Stage 4 — + GSPO

A final reinforcement-learning stage. The model reasons the most here — exploring more than one route before settling on an answer.


In [ ]:
show(runs, 'Pretrained + SFT + MPO + GSPO')

## 4. Budget Forcing

_Budget forcing_ is a blunt but effective lever: rather than letting the model decide when to stop thinking, we cut it short or push it to keep going. The s1 recipe edits the thinking stream directly — append the end-of-thinking token to stop early, or strip it and inject "Wait" to think longer.


### 4.1 Setup

We use a small open reasoning model `DeepSeek-R1-Distill-Qwen-1.5B` so whole demo can run on a free Colab GPU. The trick that makes forcing possible: we generate the thinking stream in short chunks, so we can step in between chunks and edit it — cutting it off, or pushing it to continue.


In [ ]:
# Config
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
import torch
from cachesaver.models.transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# The end-of-thinking delimiter for R1-Distill is the literal "</think>".
THINK_END = "</think>"
# Token id(s) for "</think>" so we can detect/suppress it during decoding.
THINK_END_IDS = tokenizer.encode(THINK_END, add_special_tokens=False)
print("</think> encodes to token ids:", THINK_END_IDS)
print("Loaded", MODEL_ID)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

</think> encodes to token ids: [151649]
Loaded deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B


In [ ]:
def build_prompt(question: str) -> str:
    """R1-Distill chat format. DeepSeek recommends no system prompt and forcing the
    response to begin with <think> so the model actually reasons."""
    messages = [{"role": "user", "content": question +
                 "\nPlease reason step by step, and put your final answer within \\boxed{}."}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text + "<think>\n"   # force entry into the thinking phase


def budget_forced_generate(question,
                           max_thinking_tokens=None,   # cap (None = unlimited)
                           min_thinking_tokens=0,      # floor (0 = no forced extension)
                           num_waits=0,                # max number of "Wait" injections
                           chunk=64,                   # tokens generated per step
                           max_answer_tokens=256,
                           temperature=0.6):
    """Generate with optional budget forcing. Returns dict with the trace + metadata.

    Mechanism:
      - Generate the <think> span in chunks.
      - If </think> appears but we're under min_thinking_tokens and have waits left,
        strip it and append 'Wait' (FORCE MORE).
      - If we hit max_thinking_tokens first, append '</think>\n\n**Final Answer:**'
        (FORCE LESS), then generate the answer.
    """
    import torch
    text = build_prompt(question)
    thinking_tokens = 0
    waits_used = 0
    ended_naturally = False
    forced_stop = False

    while True:
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=chunk, do_sample=True,
                temperature=temperature, top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                    skip_special_tokens=True)
        text += new_text
        thinking_tokens += chunk

        # Did the model try to end thinking?
        if THINK_END in text:
            # FORCE MORE: under the floor and waits remaining -> suppress + "Wait"
            if thinking_tokens < min_thinking_tokens and waits_used < num_waits:
                text = text.split(THINK_END)[0].rstrip() + "\nWait, "
                waits_used += 1
                continue
            ended_naturally = True
            break

        # FORCE LESS: over the cap -> inject end-of-thinking + answer cue
        if max_thinking_tokens is not None and thinking_tokens >= max_thinking_tokens:
            text = text.rstrip() + "\n" + THINK_END + "\n\n**Final Answer:**\n\n"
            forced_stop = True
            break

        # Safety valve so the loop can't run forever in a demo
        if thinking_tokens >= 8192:
            text = text.rstrip() + "\n" + THINK_END + "\n\n**Final Answer:**\n\n"
            forced_stop = True
            break

    # Generate the answer portion after thinking has ended (naturally or forced)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_answer_tokens, do_sample=True,
            temperature=temperature, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    full = text + answer

    return {
        "full_text": full,
        "thinking_tokens_generated": thinking_tokens,
        "waits_used": waits_used,
        "ended_naturally": ended_naturally,
        "forced_stop": forced_stop,
        "answer_tail": answer,
    }


### 4.2 Easy problem, unlimited budget

A trivial question (`17 + 25`) with no cap on thinking. Left to its own devices, the model spends a long chain of thought on something it could answer immediately.

Insights:

- With unlimited budget the model _overthinks_: many thinking tokens for a one-step problem.
- More thinking isn't free — it is latency and tokens spent with no gain in accuracy here.


In [ ]:
EASY_Q = "What is 17 + 25?"

r1 = budget_forced_generate(EASY_Q, max_thinking_tokens=None)  # unlimited
print("Thinking tokens generated:", r1["thinking_tokens_generated"])
print("Boxed answer:", extract_boxed(r1["full_text"]))
print(f"---Response---")
print(r1["full_text"])

Thinking tokens generated: 1472
Boxed answer: 42
---Response---
<｜begin▁of▁sentence｜><｜User｜>What is 17 + 25?
Please reason step by step, and put your final answer within \boxed{}.<｜Assistant｜><think>
<think>
Okay, so I need to figure out what 17 plus 25 is. Hmm, let me think about how to approach this. I remember that adding two numbers together involves combining their values. Maybe I can break down the numbers to make the addition easier. 

Let's see, 17 and 25. Both of these numbers are two-digit numbers, which might make it a bit tricky if I try to add them directly. Wait, no, actually, they are both two-digit, but 17 is smaller than 25. Maybe I can start by adding the ones place and then the tens place. 

So, 17 has a 7 in the ones place and a 1 in the tens place. 25 has a 5 in the ones place and a 2 in the tens place. If I add the ones place first, that would be 7 plus 5. Let me do that: 7 plus 5 is 12. Hmm, okay, so I have 12 in the ones place. But wait, 12 is more than 10, so 

### 4.3 Same easy problem, forced budget

Now we _force less_: cap thinking at zero tokens, so the model is pushed straight to its answer.

Insights:

- Cutting the thinking short still lands the correct answer — at a fraction of the tokens.
- For an easy problem, that long chain of thought was wasted budget.


In [ ]:
r2 = budget_forced_generate(EASY_Q, max_thinking_tokens=64)
print("Thinking tokens generated:", r2["thinking_tokens_generated"],
        "(forced_stop =", r2["forced_stop"], ")")
print("Boxed answer:", extract_boxed(r2["full_text"]))
print(f"---Response---")
print(r2["full_text"])

Thinking tokens generated: 64 (forced_stop = True )
Boxed answer: 42
---Response---
<｜begin▁of▁sentence｜><｜User｜>What is 17 + 25?
Please reason step by step, and put your final answer within \boxed{}.<｜Assistant｜><think>
<think>
Alright, so I need to figure out what 17 plus 25 is. Hmm, okay, let me think about how to approach this. I remember when I was learning addition, sometimes it helps to break numbers down into smaller parts to make the addition easier. Maybe I can use that method here.

First
</think>

**Final Answer:**

17 + 25 = 42

Step-by-step explanation:

1. Start with the two numbers you need to add: 17 and 25.
2. Add the units place: 7 + 5 = 12. Write down the 2 and carry over the 1.
3. Add the tens place: 1 + 2 = 3, then add the carried over 1, which makes 4.
4. Combine the results: 42.

So, 17 + 25 equals \boxed{42}.


### 4.4 Hard problem, extended thinking

A genuinely harder problem. We compare a short, unforced attempt against one where we _force more_, stripping the stop token and injecting "Wait" so the model keeps reasoning.

Insights:

- Left short, the model tends to stop early and miss the answer.
- Forcing it to keep thinking (the "Wait" injections) gives it room to recover and reach the correct result.


In [ ]:
HARD_Q = "Let f(x) = x^2 - 6x + 5. For how many integer values of x is f(x) negative?"


# First: short/no extension (likely to stop early)
short = budget_forced_generate(HARD_Q, max_thinking_tokens=256)
# Then: forced extension with Wait injections
extended = budget_forced_generate(HARD_Q, min_thinking_tokens=1500, num_waits=4)

print("SHORT  -> thinking:", short["thinking_tokens_generated"],
        "| answer:", extract_boxed(short["full_text"]))
print("EXTEND -> thinking:", extended["thinking_tokens_generated"],
        "| waits:", extended["waits_used"],
        "| answer:", extract_boxed(extended["full_text"]))
print("\n(Correct answer: f(x)<0 for 1<x<5 -> x in {2,3,4} -> 3 values.)")

SHORT  -> thinking: 256 | answer: 4
EXTEND -> thinking: 1984 | waits: 2 | answer: 3

(Correct answer: f(x)<0 for 1<x<5 -> x in {2,3,4} -> 3 values.)


# 5. Multilinguality in LLM reasoning

Improving reasoning isn't only about quality; it's about **accessibility** too: does that same quality reach everyone, whatever language they ask in?

Does a model actually _reason_ in the language you ask it in? We follow one MGSM problem across languages and model types.A plain chat model, a SOTA reasoning model's real traces, and a live thinking-vs-output view. First we set up the model and a quick English baseline.


In [ ]:
llm = LLM()
print('Model:', llm.model)
print('Languages:', ', '.join(M.LANGUAGES))

In [ ]:
PROBLEM = 5   # which MGSM problem (0–249); the same index is the same problem in every language

prob_en = M.load_mgsm('en', n=PROBLEM + 1)[PROBLEM]
r_en = await M.asolve_direct(llm, prob_en.question, 'en')

print('REQUEST:')
print(r_en.detail['prompt'])
print('\nRESPONSE:')
print(r_en.text)
ok = 'correct' if M.is_correct(r_en.answer, prob_en.answer) else 'wrong'
print(f'\n[extracted: {r_en.answer}  |  gold: {prob_en.answer}  |  {ok}]')

## 5.1 Solving MGSM

MGSM was composed by manually translating the 250 samples of GSM8K in ten different languages. Let's have a look at a couple of them.

Insights:

- The model responds in the language it was asked
- The model's performance degrades.


In [ ]:
langs = ['fr', 'zh']   # French, Russian, Chinese
probs = {lang: M.load_mgsm(lang, n=PROBLEM + 1)[PROBLEM] for lang in langs}

# solve the same problem in each language, concurrently
results = await asyncio.gather(*(M.asolve_direct(llm, probs[l].question, l) for l in langs))

# print the full request (prompt sent) and response (model output) per language
for lang, r in zip(langs, results):
    name = M.LANGUAGES[lang]['name']
    ok = 'correct' if M.is_correct(r.answer, probs[lang].answer) else 'wrong'
    print(f'========== {name} ({lang}) ==========')
    print('REQUEST:')
    print(r.detail['prompt'])
    print('\nRESPONSE:')
    print(r.text)
    print(f'\n[extracted: {r.answer}  |  gold: {probs[lang].answer}  |  {ok}]\n')

### 5.2 Using a SOTA model

In this section we're using a specialized model which in math datasets competes and overcomes state-of-the-art models such as OpenAI's o1 and DeepSeek-R1.

Insights:

- The model no longer responds in the language it was asked
- The model performs quite well in the foreign languages but not as well as in English


In [ ]:
SIZE, BUDGET = '32B', 8000   # which s1.1 model + max thinking budget to load

# load the real s1.1 trace for the same PROBLEM in every language
s1_traces = {lang: T.load_trace(SIZE, lang, BUDGET, doc_id=PROBLEM) for lang in langs}

for lang in langs:
    tr = s1_traces[lang]
    ok = 'correct' if tr.correct else 'WRONG'
    print(f'========== {M.LANGUAGES[lang]["name"]} ({lang}) — s1.1-{SIZE} ==========')
    print('REQUEST:')
    print(M.build_prompt(tr.question, lang))
    print('\nRESPONSE (s1.1 reasoning):')
    print(tr.cot)
    print(f'\n[extracted: {tr.pred}  |  gold: {tr.gold}  |  {ok}]\n')

### 5.3 Inside look

Let's have a look at a particular trace where the model is asked the question in Russian and tries to respond in English.

Insights:

- The chain of thought is in English, but the final answer is then translated into Russian
- The model burns reasoning budget on grammar not math (thinks about Russian, not in it)


In [ ]:
tr = T.load_trace('32B', 'ru', budget=8000, doc_id=5)
print('Russian problem:', tr.question[:90], '...')
print('gold:', tr.gold, ' s1 answer:', tr.pred, ' correct:', tr.correct)
display(HTML(T.highlight_html(tr.cot, title='s1.1-32B reasoning on a Russian MGSM problem')))

### 5.4 Using a reasoning model

Let's investigate a particular case of a reasoning model and specifically its **thinking** and **output** processes as two separate channels.

Insights:

- With the Spanish problem, the model thinks in English and responds in Spanish
- With the Bengali problem, the model does no thinking and directly responds in Bengali
- With the Chinese problem, the model thinks and responds in Chinese


In [ ]:
MODELS = [
    "deepseek/deepseek-v4-flash"
]
COMPARE_LANGS = ['es', 'bn', 'zh']   # languages to compare
MAX_TOKENS = 4096                    # high enough for reasoning models to finish thinking

clients = {m: LLM(model=m) for m in MODELS}
qs = {l: M.load_mgsm(l, n=PROBLEM + 1)[PROBLEM] for l in COMPARE_LANGS}

# run every (language, model) pair concurrently; tolerate per-model failures
async def _run(lang, m):
    try:
        r = await M.asolve_direct(clients[m], qs[lang].question, lang, max_tokens=MAX_TOKENS)
        return lang, m, r, None
    except Exception as e:
        return lang, m, None, f'{type(e).__name__}: {e}'

pairs = await asyncio.gather(*(_run(l, m) for l in COMPARE_LANGS for m in MODELS))
grid = {(l, m): (r, err) for l, m, r, err in pairs}

# grouped first by language, then by model — print THINKING and OUTPUT text + their token counts
for lang in COMPARE_LANGS:
    name = M.LANGUAGES[lang]['name']
    print(f'################### {name} ({lang})   (gold: {qs[lang].answer}) ###################')
    print('REQUEST:')
    print(M.build_prompt(qs[lang].question, lang))
    print()
    for m in MODELS:
        r, err = grid[(lang, m)]
        print(f'----- {m} -----')
        if err:
            print(f'[ERROR: {err}]\n')
            continue
        u = r.usage
        thinking = r.detail.get('reasoning', '')
        output = r.detail.get('content', '') or r.text
        print(f'THINKING:')
        print(thinking if thinking else '[none exposed by this model]')
        print(f'\nOUTPUT:')
        print(output if output else '[empty response]')
        ok = 'correct' if M.is_correct(r.answer, qs[lang].answer) else 'WRONG'
        print(f'\n[extracted: {r.answer}  |  {ok}]\n')
    print()

## Sources

- s1: Simple Test-Time Scaling — arXiv:2501.19393 (budget forcing, "Wait" ablation)
- ThinkPrune — arXiv:2504.01296 (R1-Distill budget-forcing implementation details)
- Follow the Path — arXiv:2505.11140 (budget sweep; ~2048-token plateau)
- Certainty-Guided Reasoning — arXiv:2509.07820 (forcing can degrade on larger models)
- DeepSeek-R1 Thoughtology — arXiv:2504.07128 (thinking-budget adherence)
- vLLM reasoning_budget PR — github.com/vllm-project/vllm/pull/37112
